# EDA 05 — Leakage Discovery
**Critical finding:** Annual_Rounds and Months_In_Season mathematically encode Target_Days.
Using these columns would make the model cheat — it would not be predicting, just calculating.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")

print("leakage Check")
print("\nHypothesis: Months_In_Season x 30 / Annual_Rounds = Target_Days")
print()
computed = df24['Months_In_Season'] * 30 / df24['Annual_Rounds']
diff = (computed - df24['Target_Days']).abs()
print(f"Max absolute difference: {diff.max():.6f} days")
print(f"Mean absolute difference: {diff.mean():.6f} days")
print()
print("Conclusion: The formula reconstructs Target_Days with near-perfect accuracy.")
print("These columns must be removed before any modelling.")


In [ ]:
# Show side by side
comparison = pd.DataFrame({
    'Annual_Rounds': df24['Annual_Rounds'],
    'Months_In_Season': df24['Months_In_Season'],
    'Target_Days_actual': df24['Target_Days'],
    'Target_Days_computed': (df24['Months_In_Season'] * 30 / df24['Annual_Rounds']).round(4),
})
print("Computed and actual (first 10 rows)")
print(comparison.head(10).to_string(index=False))


In [ ]:
# Months_In_Season scale shift
print("Months_In_Season unique values")
print(f"2024 unique: {df24['Months_In_Season'].unique()}")
print(f"2025 unique: {df25['Months_In_Season'].unique()}")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")
print(f"2026 unique: {df26['Months_In_Season'].unique()}")
print()
print("Problem found: 2026 test has Months_In_Season=5 while train data has 12.")
print("This would break any model that uses this column.")


In [ ]:
# What would happen if we naively used Annual_Rounds
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X_leak = df24[['Annual_Rounds', 'Months_In_Season']].values
y = df24['Target_Days'].values
model = LinearRegression().fit(X_leak, y)
preds = model.predict(X_leak)
print("if we naively used the leaky columns...")
print(f"R2  = {r2_score(y, preds):.4f}  ← This is essentially 1.0. Cheating.")
print(f"MAE = {mean_absolute_error(y, preds):.4f} days")
print()
print("This would be a meaningless model. Action: DROP Annual_Rounds and Months_In_Season.")


## Critical Finding
- ` Months_In_Season x 30 / Annual_Rounds  = Target_Days` with max error of ~0.0001 days
- Using these columns gives R²≈1.0 — but this is mathematical cheating, not prediction
- `Months_In_Season` also has a scale shift: train=12, test 2026=5 — would break the model
- **Action: Both columns added to the LEAK list and removed from all preprocessing**